# End-to-end ESM observation ETL and r-GCN training

This notebook demonstrates the missing ETL step: generate/load the aircraft radar KG in Neo4j, generate synthetic ESM observations, compare observations against KG radar modes, write `EvidenceEntity` nodes and candidate edges, then train the r-GCN on those evidence nodes.

Prerequisite: a local Neo4j 5 instance, for example `docker run --rm --name aircraft-kg-neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5`.

In [ ]:
from pathlib import Path
import json
import os
import sys
import yaml

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "kg_generator.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
GENERATED = REPO_ROOT / "generated"
GENERATED.mkdir(exist_ok=True)

## 1. Build and load the aircraft/radar KG

This mirrors `neo4j_kg_creation.ipynb`: constraints are created, typed KG nodes are merged, and relationships are merged by KG `id`.

In [ ]:
from neo4j import GraphDatabase
from kg_generator import generate_graph, write_json, write_triples

graph = generate_graph()
write_json(graph, GENERATED / "aircraft_radar_kg.json")
write_triples(graph, GENERATED / "aircraft_radar_triples.csv")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def wipe_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")

def create_constraints(tx):
    for statement in [
        "CREATE CONSTRAINT aircraft_family_id IF NOT EXISTS FOR (n:AircraftFamily) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT aircraft_variant_id IF NOT EXISTS FOR (n:AircraftVariant) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT radar_id IF NOT EXISTS FOR (n:Radar) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT radar_mode_id IF NOT EXISTS FOR (n:RadarMode) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT operator_id IF NOT EXISTS FOR (n:Operator) REQUIRE n.id IS UNIQUE",
    ]:
        tx.run(statement)

def load_nodes(tx, label, rows):
    tx.run(f"""
    UNWIND $rows AS row
    MERGE (n:{label} {{id: row.id}})
    SET n += row.properties
    SET n.id = row.id, n.label = row.label
    """, rows=rows)

def load_edges(tx, relation, rows):
    tx.run(f"""
    UNWIND $rows AS row
    MATCH (source {{id: row.source}})
    MATCH (target {{id: row.target}})
    MERGE (source)-[:{relation}]->(target)
    """, rows=rows)

with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(wipe_database)
    session.execute_write(create_constraints)
    labels = sorted({node["label"] for node in graph["nodes"]})
    relations = sorted({edge["relation"] for edge in graph["edges"]})
    for label in labels:
        session.execute_write(load_nodes, label, [node for node in graph["nodes"] if node["label"] == label])
    for relation in relations:
        session.execute_write(load_edges, relation, [edge for edge in graph["edges"] if edge["relation"] == relation])

len(graph["nodes"]), len(graph["edges"])

## 2. Generate observations

The observations are sampled from the same KG radar-mode bounds, but the ETL treats them as input records and re-scores them against the KG in Neo4j.

In [ ]:
from datetime import UTC, datetime
from esm_observation_generator import generate_observations

observations_doc = generate_observations(25, seed=7, start=datetime(2025, 1, 1, tzinfo=UTC), end=datetime(2025, 2, 1, tzinfo=UTC))
obs_path = GENERATED / "esm_observations_etl_demo.json"
obs_path.write_text(json.dumps(observations_doc, indent=2), encoding="utf-8")
observations_doc["observations"][0]

## 3. Compare observations to KG and create EvidenceEntity graph

`ObservationNeo4jETL` fetches `(Radar)-[:HAS_MODE]->(RadarMode)` rows with optional aircraft/operator context, scores interval overlap and categorical matches, writes observation and candidate `EvidenceEntity` nodes, and creates `HAS_CANDIDATE`, `GROUND_TRUTH_CANDIDATE`, and `SHARES_BEST_MODE` relationships.

In [ ]:
from rgcn_fusion.observation_etl import ObservationNeo4jETL, load_observations

etl = ObservationNeo4jETL(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
try:
    etl_result = etl.ingest(load_observations(obs_path), max_candidates=5)
finally:
    etl.close()
etl_result

## 4. Inspect evidence nodes and candidate edges

In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run("""
    MATCH (obs:Observation)-[r:HAS_CANDIDATE]->(candidate:CandidateEvidence)
    RETURN obs.observation_id AS observation_id, r.rank AS rank, r.score AS score,
           candidate.mode_id AS mode_id, candidate.aircraft_id AS aircraft_id
    ORDER BY observation_id, rank
    LIMIT 10
    """).data()
rows

## 5. Train the r-GCN on the EvidenceEntity subgraph

The training config points `Neo4jGraphLoader` at `EvidenceEntity` nodes. The model uses the ETL-produced `degree_score`, `text_score`, and `recency_score` features plus the ETL-produced DS masses. Classification is disabled here because the default model requires one singleton hypothesis per classification class; the ETL still stores `radar_id`, `mode_id`, `aircraft_id`, and `operator` labels for downstream experiments.

This run intentionally mirrors `rgcn_dempster_shafer_identification_demo.ipynb`: the neural net uses a seeded 50%/30%/20% train/test/validation split, writes TensorBoard progress diagnostics, and checkpoints the best validation-loss model before reporting train/test metrics.


In [ ]:
from rgcn_fusion.train import train_model

SEED = 42
TRAIN_FRACTION = 0.5
TEST_FRACTION = 0.3
VAL_FRACTION = 0.2
assert abs(TRAIN_FRACTION + TEST_FRACTION + VAL_FRACTION - 1.0) < 1e-9

train_config = {
    "neo4j": {"uri": NEO4J_URI, "user": NEO4J_USER, "password": NEO4J_PASSWORD, "database": NEO4J_DATABASE},
    "data": {
        "hypotheses": ["non_match", "match"],
        "feature_properties": ["degree_score", "text_score", "recency_score"],
        "label_property": "ds_masses",
        "classification": False,
        "node_query": """
        MATCH (n:EvidenceEntity)
        RETURN elementId(n) AS id, properties(n) AS props
        ORDER BY id
        """,
        "edge_query": """
        MATCH (s:EvidenceEntity)-[r]->(t:EvidenceEntity)
        RETURN elementId(s) AS source, elementId(t) AS target, type(r) AS type
        ORDER BY source, target, type
        """,
    },
    "model": {"hidden_features": 32, "dropout": 0.1},
    "training": {
        "epochs": 20,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "device": "cpu",
        "seed": SEED,
        "train_fraction": TRAIN_FRACTION,
        "test_fraction": TEST_FRACTION,
        "val_fraction": VAL_FRACTION,
        "patience": 30,
    },
    "output": {"directory": str(REPO_ROOT / "artifacts" / "observation_etl_demo")},
}
result = train_model(train_config)
result

## 6. Inspect model output

In [ ]:
output_path = Path(train_config["output"]["directory"]) / "node_evidence.json"
node_evidence = json.loads(output_path.read_text(encoding="utf-8"))
node_evidence[:3]

In [ ]:
driver.close()